# Comparing RC and generalized two terminal SSN linear circuit simulations in SP Ph1 domain
### Comparing SP domain simulations of Ph1 linear circuits built from Resistive Companion (RC) component models against the same linear circuits build from generalized two terminal SSN (State Space Nodal) component models.

## Run C++ example: RLC circuit

In [ ]:
import os
import subprocess

# %matplotlib widget

name = "SP_Ph1_General2TerminalSSN"

dpsim_path = (
    subprocess.Popen(["git", "rev-parse", "--show-toplevel"], stdout=subprocess.PIPE)
    .communicate()[0]
    .rstrip()
    .decode("utf-8")
)

path_exec = dpsim_path + "/build/dpsim/examples/cxx/"
sim = subprocess.Popen(
    [path_exec + name], stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(sim.communicate()[0].decode())

### Load Results

In [ ]:
from villas.dataprocessing.readtools import *
from villas.dataprocessing.timeseries import *
from villas.dataprocessing.timeseries import TimeSeries as ts
import matplotlib.pyplot as plt
import re
import numpy as np
import math

name = "SP_Ph1_VS_rlc"

work_dir = os.getcwd() + "/logs/"
path_logfile_VS_rlc_RC = work_dir + name + "_RC/" + name + "_RC" + ".csv"
ts_SP_Ph1_VS_rlc_RC = read_timeseries_dpsim(path_logfile_VS_rlc_RC)

path_logfile_VS_rlc_SSNsingleComps = (
    work_dir + name + "_SSNsingleComps/" + name + "_SSNsingleComps" + ".csv"
)
ts_SP_VS_rlc_SSNsingleComps = read_timeseries_dpsim(path_logfile_VS_rlc_SSNsingleComps)

path_logfile_VS_RLC_genSSNcombined = (
    work_dir + name + "_genSSNcombined/" + name + "_genSSNcombined" + ".csv"
)
ts_SP_Ph1_VS_RLC_genSSNcombined = read_timeseries_dpsim(
    path_logfile_VS_RLC_genSSNcombined
)

## Plot results

In [ ]:
plt.close("all")
fig1 = plt.figure()

plt.plot(
    ts_SP_Ph1_VS_rlc_RC["i12"].time,
    np.abs(ts_SP_Ph1_VS_rlc_RC["i12"].values),
    "cx",
    markevery=11,
    label="i_rlc_RC",
)
plt.plot(
    ts_SP_VS_rlc_SSNsingleComps["i12"].time,
    np.abs(ts_SP_VS_rlc_SSNsingleComps["i12"].values),
    "bx",
    markevery=13,
    label="i_rlc_SSN_single",
)

plt.plot(
    ts_SP_Ph1_VS_RLC_genSSNcombined["i12"].time,
    np.abs(ts_SP_Ph1_VS_RLC_genSSNcombined["i12"].values),
    "rx",
    markevery=17,
    label="i_rlc_SSN_combined",
)


plt.legend(loc=4)

plt.title("RC vs. single component SSN vs. generalized SSN simulation: RLC current")
plt.xlabel("t [s]")
plt.ylabel("RLC Current [A]")

## Assert

In [ ]:
epsilon = 1e-12

# compare resistive companion vs. single general SSN component modeling
assert (
    np.max(
        np.abs(
            ts_SP_Ph1_VS_rlc_RC["i12"].values
            - ts_SP_VS_rlc_SSNsingleComps["i12"].values
        )
    )
    < epsilon
)

# compare resistive companion vs. combined general SSN component modeling
assert (
    np.max(
        np.abs(
            ts_SP_Ph1_VS_rlc_RC["i12"].values
            - ts_SP_Ph1_VS_RLC_genSSNcombined["i12"].values
        )
    )
    < epsilon
)

# compare single general SSN component modeling vs. combined general SSN component modeling
assert (
    np.max(
        np.abs(
            ts_SP_Ph1_VS_RLC_genSSNcombined["i12"].values
            - ts_SP_VS_rlc_SSNsingleComps["i12"].values
        )
    )
    < epsilon
)